# **Inteligência Artificial Aplicada - 2026/1**

## **Trabalho 1: Regressão Linear com Múltiplas Variáveis**

**Professor:** Cícero Ferreira Fernandes Costa Filho  
**Aluno(a):** Maria Giovanna Gonçalves Sales - 22251138

Este notebook contém a implementação da solução para o Trabalho 1 da disciplina Inteligência Artificial Aplicada.  

O objetivo é prever a pressão arterial máxima de pacientes a partir de quatro variáveis, utilizando **Regressão Linear Múltipla** com validação cruzada de 5 pastas (*5-fold cross-validation*).

Dois métodos são implementados e comparados:
1. **scikit-learn**, com a função de alto nível `LinearRegression`
2. **Pseudo-inversa**, solução analítica implementada manualmente com NumPy

## **Montagem do Google Drive**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Importação das Bibliotecas**


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

## **Carregamento da Base de Dados**

A base de dados utilizada é o dataset **hospital** do MATLAB, disponibilizado em formato `.xls`.  
Salvamos uma cópia da base em CSV.


In [ ]:
# IMPORTANTE!!! caso o caminho fornecido abaixo não funcione, altere para apontar para onde o arquivo hospital.xls está

df_excel = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Curso IA Aplicada_2026/Bases de Dados/hospital.xls')  # lê o arquivo excel e cria um dataframe
df_excel.to_csv('hospital.csv', index=False) # salva como csv sem incluir o index das linhas como coluna extra

O dataset contém 100 linhas (pacientes) e 12 colunas, entre elas:
- `ID`, `LastName`: identificadores do paciente
- `Sex`: sexo do paciente (`'Male'` / `'Female'`)
- `Age`: idade em anos
- `Weight`: peso em libras
- `Smoker`: se o paciente é fumante (`True` / `False`)
- `BloodPressure_1`: pressão arterial sistólica (variável alvo)
- `BloodPressure_2`, `Trials_1` a `Trials_4`: outras medições


In [ ]:
df = pd.read_csv('hospital.csv')
df

,ID,LastName,Sex,Age,Weight,Smoker,BloodPressure_1,BloodPressure_2,Trials_1,Trials_2,Trials_3,Trials_4
0,YPL-320,SMITH,Male,38,176,True,124,93,18.0,NaN,NaN,NaN
1,GLI-532,JOHNSON,Male,43,163,False,109,77,11.0,13.0,22.0,NaN
2,PNI-258,WILLIAMS,Female,38,131,False,125,83,NaN,NaN,NaN,NaN
3,MIJ-579,JONES,Female,40,133,False,117,75,6.0,12.0,NaN,NaN
4,XLK-030,BROWN,Female,49,119,False,122,80,14.0,23.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,REV-997,ALEXANDER,Male,25,171,True,128,99,1.0,NaN,NaN,NaN
96,HVR-372,RUSSELL,Male,44,188,True,124,92,22.0,NaN,NaN,NaN
97,MEZ-469,GRIFFIN,Male,49,186,False,119,74,9.0,NaN,NaN,NaN
98,BEZ-311,DIAZ,Male,45,172,True,136,93,NaN,NaN,NaN,NaN


## **Pré-processamento**

### Seleção das colunas relevantes

De acordo com o enunciado, devemos usar **4 variáveis preditoras** e **1 variável alvo**:

| Coluna | Tipo | Papel no modelo |
|---|---|---|
| `Sex` | Categórica | Preditor (x₁) |
| `Age` | Numérica | Preditor (x₂) |
| `Weight` | Numérica | Preditor (x₃) |
| `Smoker` | Booleana | Preditor (x₄) |
| `BloodPressure_1` | Numérica | **Alvo (y)** |

Descartamos todas as outras colunas.


In [ ]:
colunas_desejadas = ['Sex', 'Age', 'Weight', 'Smoker', 'BloodPressure_1']
df = df[colunas_desejadas]

### Codificação da variável categórica `Sex`

Algoritmos de aprendizado de máquina trabalham apenas com números, então vamos converter essa coluna para valores numéricos (codificação binária/label encoding):

- `'Male'` → `0`
- `'Female'` → `1`

### Remoção de valores ausentes (NaN)

Isso garante que o modelo não receba entradas inválidas durante o treinamento.  
As colunas selecionadas não possuem valores ausentes, mas faremos o passo por questões de boas práticas.

In [ ]:
# converte sex para número
df.loc[df['Sex'] == 'Female', 'Sex'] = 1
df.loc[df['Sex'] == 'Male', 'Sex'] = 0

# remove valores ausentes
df = df.dropna()

df

,Sex,Age,Weight,Smoker,BloodPressure_1
0,0,38,176,True,124
1,0,43,163,False,109
2,1,38,131,False,125
3,1,40,133,False,117
4,1,49,119,False,122
...,...,...,...,...,...
95,0,25,171,True,128
96,0,44,188,True,124
97,0,49,186,False,119
98,0,45,172,True,136


## **Separação entre Preditores (X) e Alvo (y)**

**X:** matriz de features  
**y:** vetor alvo

In [ ]:
X = df[['Sex', 'Age', 'Weight', 'Smoker']].values.astype(float) # .values converte o dataframe pandas para um array numpy puro

y = df['BloodPressure_1'].values.astype(float)

print(f"Shape de X: {X.shape}")  # esperado: (100, 4)
print(f"Shape de y: {y.shape}")  # esperado: (100,)

Shape de X: (100, 4)
Shape de y: (100,)


In [ ]:
X

array([[  0.,  38., 176.,   1.],
       [  0.,  43., 163.,   0.],
       [  1.,  38., 131.,   0.],
       [  1.,  40., 133.,   0.],
       [  1.,  49., 119.,   0.],
       [  1.,  46., 142.,   0.],
       [  1.,  33., 142.,   1.],
       [  0.,  40., 180.,   0.],
       [  0.,  28., 183.,   0.],
       [  1.,  31., 132.,   0.],
       [  1.,  45., 128.,   0.],
       [  1.,  42., 137.,   0.],
       [  0.,  25., 174.,   0.],
       [  0.,  39., 202.,   1.],
       [  1.,  36., 129.,   0.],
       [  0.,  48., 181.,   1.],
       [  0.,  32., 191.,   1.],
       [  1.,  27., 131.,   1.],
       [  0.,  37., 179.,   0.],
       [  0.,  50., 172.,   0.],
       [  1.,  48., 133.,   0.],
       [  1.,  39., 117.,   0.],
       [  1.,  41., 137.,   0.],
       [  1.,  44., 146.,   1.],
       [  1.,  28., 123.,   1.],
       [  0.,  25., 189.,   0.],
       [  1.,  39., 143.,   0.],
       [  1.,  25., 114.,   0.],
       [  0.,  36., 166.,   0.],
       [  0.,  30., 186.,   1.],
       [  

In [ ]:
y

array([124., 109., 125., 117., 122., 121., 130., 115., 115., 118., 114.,
       115., 127., 130., 114., 130., 124., 123., 119., 125., 121., 123.,
       114., 128., 129., 114., 113., 125., 120., 127., 134., 121., 115.,
       127., 121., 127., 136., 117., 124., 120., 128., 116., 132., 137.,
       117., 116., 119., 123., 116., 124., 129., 130., 132., 117., 129.,
       118., 120., 138., 117., 113., 122., 115., 120., 117., 123., 123.,
       119., 110., 121., 138., 125., 122., 120., 117., 125., 124., 121.,
       118., 120., 118., 118., 122., 134., 131., 113., 125., 135., 128.,
       123., 122., 138., 124., 130., 123., 129., 128., 124., 119., 136.,
       114.])

In [ ]:
# fazer teste usando normalização

# scaler = preprocessing.MinMaxScaler()
# X_scaled = scaler.fit_transform(X)

## **Configuração da Validação Cruzada (K-Fold)**

### **Parâmetros usados**

```python
KFold(n_splits=5, shuffle=True, random_state=42)
```

| Parâmetro | Valor | Significado |
|---|---|---|
| `n_splits` | `5` | Divide os dados em 5 partes de ≈ 20 amostras cada |
| `shuffle` | `True` | Embaralha os dados antes de dividir (evita viés de ordenação) |
| `random_state` | `42` | Semente aleatória fixa → garante sempre os mesmos resultados |

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

## **Implementação do Método 1: Regressão Linear com scikit-learn**

In [ ]:
modelo = LinearRegression() # inicializa o modelo de regressão linear

erros_mse = [] # lista para guardar erros mse de cada pasta
coeficientes_pearson = [] # lista para guardar c. de pearson de cada pasta

numero_da_pasta = 1

for indice_treino, indice_teste in kf.split(X):
    # seleciona as amostras de treino e teste com base nos índices da pasta atual
    X_treino, X_teste = X[indice_treino], X[indice_teste]
    y_treino, y_teste = y[indice_treino], y[indice_teste]

    modelo.fit(X_treino, y_treino) # treina o modelo

    y_predito = modelo.predict(X_teste) # faz predições

    # guarda mse
    erro = mean_squared_error(y_teste, y_predito)

    # guarda c. de pearson
    pearson = pearsonr(y_teste, y_predito)[0]

    erros_mse.append(erro)
    coeficientes_pearson.append(pearson)

    print(f"Pasta {numero_da_pasta} -> Pearson: {pearson:.4f} | Erro Quadrático (MSE): {erro:.4f}")
    numero_da_pasta += 1

Pasta 1 -> Pearson: 0.5190 | Erro Quadrático (MSE): 32.4615
Pasta 2 -> Pearson: 0.6989 | Erro Quadrático (MSE): 18.2036
Pasta 3 -> Pearson: 0.7297 | Erro Quadrático (MSE): 15.1829
Pasta 4 -> Pearson: 0.8383 | Erro Quadrático (MSE): 28.3679
Pasta 5 -> Pearson: 0.6405 | Erro Quadrático (MSE): 28.6908


## **Médias Finais (scikit-learn)**


In [ ]:
media_erro = np.mean(erros_mse)
media_pearson = np.mean(coeficientes_pearson)

print("Médias finais (scikit-learn)")
print(f"Valor Médio do Coeficiente de Pearson: {media_pearson:.4f}")
print(f"Valor Médio do Erro Médio Quadrático:  {media_erro:.4f}")

Médias finais (scikit-learn)
Valor Médio do Coeficiente de Pearson: 0.6853
Valor Médio do Erro Médio Quadrático:  24.5813


## **Implementação do Método 2: Regressão Linear pela Pseudo-Inversa**

Para N amostras e p preditores, o sistema a resolver é:

$$\mathbf{y} = X \mathbf{w}$$

Como temos muito mais amostras (N=100) do que parâmetros (p=4+1=5), o sistema é sobredeterminado, ou seja, não existe solução exata.  
A solução que minimiza o erro quadrático é dada pela pseudo-inversa de Moore-Penrose:

$$\mathbf{w} = X^+ \mathbf{y} = (X^\top X)^{-1} X^\top \mathbf{y}$$

### **Bias**

A equação do modelo é:

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + k$$

O termo `k` é o bias, que permite que a reta não passe pela origem. Para incluí-lo na multiplicação de matrizes, adicionamos uma coluna extra de `1s` em X:

```
X original (100x4)  →  X com bias (100x5)
[Sex, Age, Wgt, Smk]   [Sex, Age, Wgt, Smk, 1]
[  0,  38, 176,   1]   [  0,  38, 176,   1, 1]
[  0,  43, 163,   0]   [  0,  43, 163,   0, 1]
         ...                      ...
```

Assim, o vetor de pesos w = [w1, w2, w3, w4, k] captura o intercepto naturalmente.


In [ ]:
# guardam os resultados de cada pasta
erros_mse_pi = []
coeficientes_pearson_pi = []

numero_da_pasta = 1

for indice_treino, indice_teste in kf.split(X):
    # separa treino e teste
    X_treino, X_teste = X[indice_treino], X[indice_teste]
    y_treino, y_teste = y[indice_treino], y[indice_teste]

    # adiciona coluna de 1s
    # np.ones(N) cria o vetor de 1s e column_stack empilha
    X_treino_bias = np.column_stack([X_treino, np.ones(len(X_treino))])
    X_teste_bias  = np.column_stack([X_teste,  np.ones(len(X_teste))])

    # cálculo da pseudo-inversa
    # PI = (X^T . X)^{-1} . X^T
    # X_treino_bias.T = transposta de X
    # @ = multiplicação matricial
    # np.linalg.inv = inversa da matriz quadrada (X^T . X)
    PI = np.linalg.inv(X_treino_bias.T @ X_treino_bias) @ X_treino_bias.T

    # cálculo dos pesos
    # w = PI . y
    # vai ser [w1, w2, w3, w4, k]
    w = PI @ y_treino

    # predição
    # y_predito = X_teste . w
    # multiplica cada linha de X_teste pelo vetor de pesos
    y_predito_pi = X_teste_bias @ w

    # métricas
    erro_pi    = mean_squared_error(y_teste, y_predito_pi)
    pearson_pi = pearsonr(y_teste, y_predito_pi)[0]

    erros_mse_pi.append(erro_pi)
    coeficientes_pearson_pi.append(pearson_pi)

    print(f"Pasta {numero_da_pasta} [PI] -> Pearson: {pearson_pi:.4f} | MSE: {erro_pi:.4f}")
    numero_da_pasta += 1

Pasta 1 [PI] -> Pearson: 0.5190 | MSE: 32.4615
Pasta 2 [PI] -> Pearson: 0.6989 | MSE: 18.2036
Pasta 3 [PI] -> Pearson: 0.7297 | MSE: 15.1829
Pasta 4 [PI] -> Pearson: 0.8383 | MSE: 28.3679
Pasta 5 [PI] -> Pearson: 0.6405 | MSE: 28.6908


## **Médias Finais (Pseudo-Inversa)**

In [ ]:
media_erro_pi    = np.mean(erros_mse_pi)
media_pearson_pi = np.mean(coeficientes_pearson_pi)

print("Médias finais (Pseudo-Inversa)")
print(f"Valor Médio do Coeficiente de Pearson: {media_pearson_pi:.4f}")
print(f"Valor Médio do Erro Médio Quadrático:  {media_erro_pi:.4f}")

Médias finais (Pseudo-Inversa)
Valor Médio do Coeficiente de Pearson: 0.6853
Valor Médio do Erro Médio Quadrático:  24.5813


## **Comparação entre os dois métodos**

Os dois métodos resolvem o mesmo problema -> minimização do erro quadrático.    
A diferença está na implementação:
- O scikit-learn usa Decomposição em Valores Singulares (SVD)
- A pseudo-inversa usa $(X^\top X)^{-1} X^\top$ diretamente

In [ ]:
print(f"{'Método':<20} {'Pearson Médio':>15} {'MSE Médio':>12}")
print(f"{'scikit-learn':<20} {media_pearson:>15.4f} {media_erro:>12.4f}")
print(f"{'Pseudo-Inversa':<20} {media_pearson_pi:>15.4f} {media_erro_pi:>12.4f}")

Método                 Pearson Médio    MSE Médio
scikit-learn                  0.6853      24.5813
Pseudo-Inversa                0.6853      24.5813


Podemos observar que a média das métricas para os dois métodos utilizados foram iguais.

Os dois métodos utilizados tentam minimizar a mesma função de custo, e esse esse mínimo é único (desde que X tenha colunas linearmente independentes). Logo, os pesos, as predições e as métricas são os mesmos.

No relatório fornecido, se encontram explicações mais detalhadas acerca dos resultados.